# UQRoute-TC — Qwen 1.5B repeated-sample status audit

**Scope:** Re-audit the saved 30-case development run on a CPU runtime. Each case keeps all ten recorded sample slots; a truncated or failed request receives an explicit outcome key. The audit rereads the saved raw responses and does not generate model outputs.

**Recorded evidence:** Selected case identity, request and parser status counts, token-evidence checks, canonical and exact-string clusters, and validation errors. The original records remain in Drive.


## 1. Restore the pinned parser and split


In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import json
import subprocess
import sys

drive.mount('/content/drive')
root = Path('/content/drive/MyDrive/UQRoute-TC')
output_dir = root / 'pilots/qwen25-15b-repeated-dev30-v1'
manifest = json.loads((output_dir / 'manifest.json').read_text())
benchmark_revision = 'd5d03180de41eb30a6c796d04a9bfbd9dad85c1d'
code_revision = '7c178b4c038e6af44a5c4a6599a5d3bf390e0bbf'
assert manifest['benchmark_revision'] == benchmark_revision
assert manifest['code_revision'] == code_revision
assert manifest['model_id'] == 'Qwen/Qwen2.5-1.5B-Instruct'
assert manifest['sample_seeds'] == list(range(1729, 1739))
assert manifest['temperature'] == 0.7 and manifest['top_p'] == 1.0
selected = manifest['selected_cases']
assert len(selected) == 30
selection_digest = hashlib.sha256(json.dumps(selected, sort_keys=True, separators=(',', ':')).encode()).hexdigest()
assert selection_digest == manifest['selection_digest'] == '082e150679277d7d55eaa88b0b82cb236a82a32a0d6853c2cae8dc307b41fa67'

benchmark = Path('/content/robustbench-tc-release')
if not benchmark.exists():
    subprocess.run(['git', 'clone', 'https://github.com/WillChow66/robustbench-tc-release.git', str(benchmark)], check=True)
subprocess.run(['git', '-C', str(benchmark), 'checkout', '--detach', benchmark_revision], check=True)
assert subprocess.check_output(['git', '-C', str(benchmark), 'rev-parse', 'HEAD'], text=True).strip() == benchmark_revision
project = Path('/content/uqroute-tc-repeated-check')
if not project.exists():
    subprocess.run(['git', 'clone', 'https://github.com/RavindraSSK/uqroute-tc.git', str(project)], check=True)
subprocess.run(['git', '-C', str(project), 'fetch', 'origin', 'feat/task2-canonical-parser'], check=True)
subprocess.run(['git', '-C', str(project), 'checkout', '--detach', code_revision], check=True)
assert subprocess.check_output(['git', '-C', str(project), 'rev-parse', 'HEAD'], text=True).strip() == code_revision
split_path = project / 'docs/splits/group_split_v1.json'
assert hashlib.sha256(split_path.read_bytes()).hexdigest() == manifest['split_sha256']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'openai>=1.50,<3', '-e', str(project)], check=True)
print('Saved cases:', len(selected), '| project code:', code_revision)


## 2. Recompute the status-aware clusters


In [ ]:
from collections import Counter
sys.path.insert(0, str(benchmark / 'scripts'))
sys.path.insert(0, str(project / 'src'))
from run_eval import parse_tool_calls
from uqroute_tc.inference.pilot import _atomic_json, token_evidence
from uqroute_tc.parsing.canonical import parse_prediction
from uqroute_tc.uncertainty.repeated import cluster_scores

def record_path(item, index):
    key = item['source_file'] + '\0' + item['case_id']
    folder = hashlib.sha256(key.encode()).hexdigest()
    return output_dir / 'records' / folder / f'sample_{index:02d}.json'

def status_clusters(rows, benchmark_name):
    canonical = []
    exact = []
    outcomes = []
    for row in rows:
        status, raw = row['status'], row['raw_output']
        if status in {'truncated', 'request_error'}:
            canonical.append(json.dumps([status]))
            exact.append(json.dumps([status, raw], ensure_ascii=False))
            outcomes.append(status)
        else:
            outcome = parse_prediction(raw, benchmark_name, parse_tool_calls)
            canonical.append(outcome.key)
            exact.append(json.dumps(['output', raw], ensure_ascii=False))
            outcomes.append(outcome.status)
    return cluster_scores(canonical), cluster_scores(exact), outcomes

entries = []
for item in selected:
    rows = []
    errors = []
    for index, seed in enumerate(manifest['sample_seeds']):
        path = record_path(item, index)
        if not path.exists():
            errors.append({'sample_index': index, 'reason': 'missing saved record'})
            continue
        row = json.loads(path.read_text())
        rows.append(row)
        if (row['case_id'], row['source_file'], row['seed'], row['sample_index']) != (
                item['case_id'], item['source_file'], seed, index):
            errors.append({'sample_index': index, 'reason': 'record identity differs'})
            continue
        status = row['status']
        if status not in {'complete', 'invalid_evidence', 'truncated', 'request_error'}:
            errors.append({'sample_index': index, 'reason': 'unknown generation status'})
            continue
        if status == 'request_error':
            continue
        response = row.get('response') or {}
        choices = response.get('choices') or []
        if len(choices) != 1:
            errors.append({'sample_index': index, 'reason': 'missing saved response choice'})
            continue
        choice = choices[0]
        if (choice.get('message') or {}).get('content', '') != row['raw_output']:
            errors.append({'sample_index': index, 'reason': 'visible response differs'})
            continue
        if status == 'truncated':
            if choice.get('finish_reason') != 'length':
                errors.append({'sample_index': index, 'reason': 'truncation lacks length finish reason'})
            continue
        if status == 'complete' and not token_evidence(choice)['valid']:
            errors.append({'sample_index': index, 'reason': 'chosen-token evidence invalid'})
            continue
        reparsed = parse_tool_calls(row['raw_output'], item['benchmark'])
        if json.loads(json.dumps(reparsed, ensure_ascii=False)) != row['parsed_calls']:
            errors.append({'sample_index': index, 'reason': 'saved parser calls differ after JSON round-trip'})
    status_counts = Counter(row['status'] for row in rows)
    parser_counts = Counter(row['parser_status'] for row in rows)
    entry = {**item, 'record_status_counts': dict(status_counts),
             'parser_status_counts': dict(parser_counts),
             'validation_errors': errors, 'scored': len(rows) == 10 and not errors}
    if entry['scored']:
        canonical, exact, outcomes = status_clusters(rows, item['benchmark'])
        entry.update(canonical_entropy_nats=canonical.entropy_nats,
                     canonical_disagreement=canonical.disagreement,
                     canonical_cluster_sizes=canonical.cluster_sizes,
                     exact_string_entropy_nats=exact.entropy_nats,
                     exact_string_disagreement=exact.disagreement,
                     outcome_statuses=outcomes,
                     elapsed_seconds_sum=sum(row['elapsed_seconds'] for row in rows))
    entries.append(entry)

report = {'manifest': manifest, 'cases': len(entries),
          'scored_cases': sum(entry['scored'] for entry in entries),
          'request_status_counts': dict(sum((Counter(entry['record_status_counts'])
                                              for entry in entries), Counter())),
          'validation_error_cases': sum(bool(entry['validation_errors']) for entry in entries),
          'entries': entries}
report_path = output_dir / 'repeated_dev30_status_audit.json'
_atomic_json(report_path, report)
print('Scored cases:', report['scored_cases'], '/ 30')
print('Request statuses:', report['request_status_counts'])
print('Validation error cases:', report['validation_error_cases'])
for entry in entries:
    if entry['record_status_counts'].get('truncated') or entry['validation_errors']:
        print(entry['benchmark'], entry['source_file'], entry['record_status_counts'],
              'clusters:', entry.get('canonical_cluster_sizes'),
              'errors:', entry['validation_errors'])
print('Audit:', report_path)


## 3. Export the compact audit


In [ ]:
from google.colab import files
files.download(str(report_path))
